# AstroShield — Evolucionar una constelación inteligente
**Proyecto en equipo - Ingeniería en Tecnologías de la Información**

Este notebook contiene la implementación completa del sistema de optimización de una constelación de 8 satélites utilizando un algoritmo evolutivo, restricciones espaciales y visualización 3D interactiva en Plotly.

In [1]:
import math
import random
import numpy as np
import pandas as pd
import plotly.graph_objects as go

## Sección 1: Matemáticas, Distancias y Restricciones (Integrante 1)
Aquí se implementa la fórmula de Haversine, el cálculo de cobertura de las 6 regiones y la validación de las zonas de exclusión (desechos y objetos ocupados).

In [2]:
# Datos iniciales de las regiones (Latitud, Longitud, Prioridad: 3=Alta, 2=Media, 1=Normal)
REGIONS = {
    "Yucatán": {"coords": (20.5, -89.0), "priority": 3},
    "Amazonía": {"coords": (-3.5, -62.0), "priority": 3},
    "California": {"coords": (36.8, -119.4), "priority": 2},
    "Sahel": {"coords": (15.0, 0.0), "priority": 2},
    "Japón": {"coords": (36.0, 138.0), "priority": 2},
    "Australia": {"coords": (-25.0, 133.0), "priority": 1}
}

# Zonas restringidas por desechos (Centro lat, lon, radio en grados)
DEBRIS_ZONES = [
    {"name": "Atlántico Norte", "center": (45.0, -35.0), "radius": 12.0},
    {"name": "Pacífico Sur", "center": (-30.0, -120.0), "radius": 10.0},
    {"name": "Zona polar norte", "center": (78.0, 30.0), "radius": 8.0}
]

# Objetos ocupados existentes (Centro lat, lon, radio de exclusión)
BLOCKED_OBJECTS = [
    {"name": "Objeto ocupado A", "center": (10.0, 45.0), "radius": 5.0},
    {"name": "Objeto ocupado B", "center": (-15.0, 110.0), "radius": 5.0}
]

OBSERVATION_RADIUS_KM = 300.0

def haversine(lat1, lon1, lat2, lon2):
    """Calcula la distancia en kilómetros entre dos puntos sobre la Tierra."""
    R = 6371.0  # Radio de la Tierra en km
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2)**2
    c = 2 * math.asin(math.sqrt(a))
    return R * c

def is_valid_position(lat, lon):
    """Verifica si una posición viola las zonas de desechos o los objetos ocupados."""
    # Verificar desechos espaciales
    for zone in DEBRIS_ZONES:
        dist = haversine(lat, lon, zone["center"][0], zone["center"][1])
        # Aproximación de grados a km (~111 km por grado)
        if dist <= zone["radius"] * 111:
            return False

    # Verificar objetos ocupados
    for obj in BLOCKED_OBJECTS:
        dist = haversine(lat, lon, obj["center"][0], obj["center"][1])
        if dist <= obj["radius"] * 111:
            return False

    return True

print("Módulo de matemáticas y restricciones cargado.")

Módulo de matemáticas y restricciones cargado.


## Sección 2: Algoritmo Evolutivo (Integrante 2)
Implementación de la población de 8 satélites, operadores genéticos (selección, cruce, mutación) y la función de fitness.

In [3]:
import random
import numpy as np

class AlgoritmoEvolutivoConstelacion:
    def __init__(self, evaluar_fitness_func, validar_posicion_func=None, num_satelites=8):
        """
        Inicializa el algoritmo evolutivo para la constelación AstroShield.
        """
        self.evaluar_fitness = evaluar_fitness_func
        self.validar_posicion = validar_posicion_func
        self.num_satelites = num_satelites

    def generar_individuo(self):
        """
        Genera un individuo válido (8 satélites con coordenadas [lat, lon]).
        """
        individuo = []
        intentos = 0
        while len(individuo) < self.num_satelites and intentos < 1000:
            lat = random.uniform(-80, 80)
            lon = random.uniform(-180, 180)

            if self.validar_posicion is None or self.validar_posicion(lat, lon):
                individuo.append([lat, lon])
            intentos += 1

        while len(individuo) < self.num_satelites:
            individuo.append([random.uniform(-80, 80), random.uniform(-180, 180)])

        return individuo

    def generar_poblacion_inicial(self, tamano_poblacion):
        """Genera la población inicial de configuraciones."""
        return [self.generar_individuo() for _ in range(tamano_poblacion)]

    def seleccion_torneo(self, poblacion, aptitudes, k=3):
        """Selecciona al mejor individuo mediante torneo."""
        seleccionados = random.sample(list(zip(poblacion, aptitudes)), k)
        seleccionados.sort(key=lambda x: x[1], reverse=True)
        return seleccionados[0][0]

    def cruce(self, padre1, padre2):
        """Operador de cruce de un punto entre dos configuraciones."""
        punto = random.randint(1, self.num_satelites - 1)
        hijo1 = padre1[:punto] + padre2[punto:]
        hijo2 = padre2[:punto] + padre1[punto:]
        return hijo1, hijo2

    def mutacion(self, individuo, tasa_mutacion=0.1):
        """Aplica una ligera perturbación aleatoria a las coordenadas de los satélites."""
        individuo_mutado = []
        for sat in individuo:
            if random.random() < tasa_mutacion:
                lat = max(-90, min(90, sat[0] + random.gauss(0, 3)))
                lon = max(-180, min(180, sat[1] + random.gauss(0, 3)))
                individuo_mutado.append([lat, lon])
            else:
                individuo_mutado.append(sat)
        return individuo_mutado

    def ejecutar(self, generaciones=50, tamano_poblacion=30, tasa_mutacion=0.1):
        """
        Ejecuta el ciclo principal del algoritmo evolutivo y registra el historial.
        """
        poblacion = self.generar_poblacion_inicial(tamano_poblacion)
        historial_max = []
        historial_avg = []
        mejor_individuo = None
        mejor_fitness = float('-inf')

        for gen in range(generaciones):
            aptitudes = [self.evaluar_fitness(ind) for ind in poblacion]

            max_gen = max(aptitudes)
            avg_gen = sum(aptitudes) / len(aptitudes)
            historial_max.append(max_gen)
            historial_avg.append(avg_gen)

            idx_mejor = np.argmax(aptitudes)
            if aptitudes[idx_mejor] > mejor_fitness:
                mejor_fitness = aptitudes[idx_mejor]
                mejor_individuo = poblacion[idx_mejor]

            nueva_poblacion = []
            while len(nueva_poblacion) < tamano_poblacion:
                p1 = self.seleccion_torneo(poblacion, aptitudes)
                p2 = self.seleccion_torneo(poblacion, aptitudes)
                h1, h2 = self.cruce(p1, p2)

                nueva_poblacion.append(self.mutacion(h1, tasa_mutacion))
                if len(nueva_poblacion) < tamano_poblacion:
                    nueva_poblacion.append(self.mutacion(h2, tasa_mutacion))

            poblacion = nueva_poblacion

        return mejor_individuo, mejor_fitness, historial_max, historial_avg

## Sección 3: Visualización 3D Interactiva (Integrante 3)
Generación del globo terráqueo interactivo usando Plotly para mostrar satélites, regiones y zonas de exclusión.

### Preparación de la visualización y evaluación\n\nEsta sección conecta el globo 3D con la configuración que produce el algoritmo del Integrante 2. Se reutilizan las regiones, las distancias y las restricciones definidas en la Sección 1.

In [4]:
%pip -q install plotly ipywidgets pandas numpy
from IPython.display import display, Markdown, clear_output

def angular_distance_deg(lat1, lon1, lat2, lon2):
    return math.degrees(haversine(lat1, lon1, lat2, lon2) / 6371.0)

def satellite_region_distance(sat, region_name):
    return haversine(sat[0], sat[1], *REGIONS[region_name]["coords"])

def evaluar_fitness(individuo):
    """Premia la cobertura según prioridad, incentiva la cercanía y penaliza posiciones inválidas."""
    score = 0.0
    for nombre, info in REGIONS.items():
        dist_min = min(haversine(s[0], s[1], *info["coords"]) for s in individuo)
        if dist_min <= OBSERVATION_RADIUS_KM:
            score += info["priority"]
        else:
            score += 0.25 * info["priority"] * max(0.0, 1.0 - dist_min / 3000.0)
    invalid_count = sum(not is_valid_position(s[0], s[1]) for s in individuo)
    score -= invalid_count * 2.5
    score += sum(is_valid_position(s[0], s[1]) for s in individuo) * 0.02
    return float(score)

def coverage_report(individuo):
    rows = []
    for name, info in REGIONS.items():
        distances = [haversine(s[0], s[1], *info["coords"]) for s in individuo]
        idx = int(np.argmin(distances))
        rows.append({
            "Región": name,
            "Prioridad": {3: "Alta", 2: "Media", 1: "Normal"}[info["priority"]],
            "Distancia mínima (km)": round(distances[idx], 1),
            "Satélite más cercano": idx + 1,
            "Cubierta (≤300 km)": "Sí" if distances[idx] <= OBSERVATION_RADIUS_KM else "No"
        })
    return pd.DataFrame(rows)

print("Datos, función de evaluación y reporte de cobertura listos.")


Note: you may need to restart the kernel to use updated packages.
Datos, función de evaluación y reporte de cobertura listos.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Ejecución del algoritmo evolutivo del equipo

In [5]:
algoritmo = AlgoritmoEvolutivoConstelacion(
    evaluar_fitness_func=evaluar_fitness,
    validar_posicion_func=is_valid_position,
    num_satelites=8
)
MEJOR_INDIVIDUO, MEJOR_FITNESS, HISTORIAL_MAX, HISTORIAL_AVG = algoritmo.ejecutar(
    generaciones=60, tamano_poblacion=50, tasa_mutacion=0.12
)
print(f"Mejor fitness encontrado: {MEJOR_FITNESS:.3f}")
display(pd.DataFrame([
    {"Satélite": i + 1, "Latitud": round(s[0], 3), "Longitud": round(s[1], 3),
     "Posición válida": "Sí" if is_valid_position(*s) else "No"}
    for i, s in enumerate(MEJOR_INDIVIDUO)
]))
display(coverage_report(MEJOR_INDIVIDUO))


Mejor fitness encontrado: 10.160


,Satélite,Latitud,Longitud,Posición válida
0,1,37.642,140.386,Sí
1,2,50.943,140.562,Sí
2,3,54.690,-109.878,Sí
3,4,37.587,-117.351,Sí
4,5,-24.322,30.484,Sí
5,6,18.779,-87.070,Sí
6,7,-12.357,-66.392,Sí
7,8,-3.891,-61.252,Sí


,Región,Prioridad,Distancia mínima (km),Satélite más cercano,Cubierta (≤300 km)
0,Yucatán,Alta,278.3,6,Sí
1,Amazonía,Alta,93.7,8,Sí
2,California,Media,201.5,4,Sí
3,Sahel,Media,5483.4,5,No
4,Japón,Media,280.1,1,Sí
5,Australia,Normal,7008.1,1,No


In [6]:
from IPython.display import display, HTML

def latlon_to_xyz(lat, lon, radius=1.0):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    return (radius*np.cos(lat_r)*np.cos(lon_r),
            radius*np.cos(lat_r)*np.sin(lon_r), radius*np.sin(lat_r))

def circle_on_globe(center, radius_deg, n=120, globe_radius=1.018):
    lat0, lon0 = center
    bearings = np.linspace(0, 2*np.pi, n)
    ang, lat1, lon1 = np.radians(radius_deg), np.radians(lat0), np.radians(lon0)
    lats, lons = [], []
    for b in bearings:
        lat = np.arcsin(np.sin(lat1)*np.cos(ang)+np.cos(lat1)*np.sin(ang)*np.cos(b))
        lon = lon1 + np.arctan2(np.sin(b)*np.sin(ang)*np.cos(lat1),
                                np.cos(ang)-np.sin(lat1)*np.sin(lat))
        lats.append(np.degrees(lat)); lons.append((np.degrees(lon)+180)%360-180)
    return latlon_to_xyz(np.array(lats), np.array(lons), globe_radius)

def crear_dashboard(individuo, historial_max=None, historial_avg=None):
    reporte = coverage_report(individuo)
    cubiertas = int((reporte['Cubierta (≤300 km)'] == 'Sí').sum())
    invalidos = sum(not is_valid_position(*s) for s in individuo)
    fitness = evaluar_fitness(individuo)
    tasa = 100*cubiertas/max(1, len(REGIONS))
    display(HTML(f'''<div style="font-family:Arial,sans-serif;background:linear-gradient(120deg,#07111f,#0b1e31);border:1px solid #1c4962;border-radius:16px;padding:22px;color:#e6faff;margin:4px 0 14px 0">
      <div style="display:flex;align-items:center;gap:12px"><div style="font-size:28px">🛰️</div><div><div style="font-size:11px;letter-spacing:3px;color:#00e5c3">ASTROSHIELD / MISSION CONTROL</div><div style="font-size:24px;font-weight:700;margin-top:4px">Centro de control de constelación</div><div style="font-size:12px;color:#8aa9bc;margin-top:5px">Visualización interactiva · Optimización evolutiva · Monitoreo regional</div></div><div style="margin-left:auto;border:1px solid #087f70;color:#48f3cf;padding:7px 12px;border-radius:20px;font-size:11px">● SIMULACIÓN GENERADA</div></div>
      <div style="display:grid;grid-template-columns:repeat(4,minmax(120px,1fr));gap:10px;margin-top:20px">
      <div style="background:#0c2637;border:1px solid #17445a;border-radius:10px;padding:14px"><div style="font-size:10px;color:#8daaba;letter-spacing:1px">SATÉLITES</div><div style="font-size:27px;font-weight:700;color:#f1fbff">{len(individuo)} <span style="font-size:11px;color:#00e5c3">● ACTIVO</span></div><div style="font-size:11px;color:#7896a8">Constelación optimizada</div></div>
      <div style="background:#0c2637;border:1px solid #17445a;border-radius:10px;padding:14px"><div style="font-size:10px;color:#8daaba;letter-spacing:1px">COBERTURA REGIONAL</div><div style="font-size:27px;font-weight:700;color:#00e5c3">{cubiertas}/{len(REGIONS)}</div><div style="font-size:11px;color:#7896a8">Dentro del radio de 300 km</div></div>
      <div style="background:#0c2637;border:1px solid #17445a;border-radius:10px;padding:14px"><div style="font-size:10px;color:#8daaba;letter-spacing:1px">FITNESS</div><div style="font-size:27px;font-weight:700;color:#f8ce66">{fitness:.3f}</div><div style="font-size:11px;color:#7896a8">Evaluación de la solución</div></div>
      <div style="background:#0c2637;border:1px solid #17445a;border-radius:10px;padding:14px"><div style="font-size:10px;color:#8daaba;letter-spacing:1px">POSICIONES INVÁLIDAS</div><div style="font-size:27px;font-weight:700;color:{'#ff6482' if invalidos else '#00e5c3'}">{invalidos}</div><div style="font-size:11px;color:#7896a8">Según restricciones configuradas</div></div>
      </div></div>'''))

    fig = go.Figure()
    u = np.linspace(0, 2*np.pi, 100); v = np.linspace(-np.pi/2, np.pi/2, 60)
    xx = np.outer(np.cos(u), np.cos(v)); yy = np.outer(np.sin(u), np.cos(v)); zz = np.outer(np.ones_like(u), np.sin(v))
    fig.add_trace(go.Surface(x=xx,y=yy,z=zz,surfacecolor=zz,
        colorscale=[[0,'#07172b'],[0.25,'#0b3650'],[0.55,'#11647a'],[0.8,'#238b91'],[1,'#9cc8a6']],
        cmin=-1,cmax=1,opacity=.98,showscale=False,hoverinfo='skip',name='Tierra'))
    # Grid latitude/longitude for a technical cartographic feel
    for lat in [-60,-30,0,30,60]:
        lons=np.linspace(-180,180,160); lats=np.full_like(lons,lat)
        gx,gy,gz=latlon_to_xyz(lats,lons,1.003)
        fig.add_trace(go.Scatter3d(x=gx,y=gy,z=gz,mode='lines',line=dict(color='#56c7db',width=1),opacity=.22,showlegend=False,hoverinfo='skip'))
    for lon in range(-150,181,30):
        lats=np.linspace(-89,89,140); lons=np.full_like(lats,lon)
        gx,gy,gz=latlon_to_xyz(lats,lons,1.003)
        fig.add_trace(go.Scatter3d(x=gx,y=gy,z=gz,mode='lines',line=dict(color='#56c7db',width=1),opacity=.22,showlegend=False,hoverinfo='skip'))
    # Restricted zones / occupied objects
    for idx,zone in enumerate(DEBRIS_ZONES+BLOCKED_OBJECTS):
        x,y,z=circle_on_globe(zone['center'],zone['radius'])
        label=zone.get('name',f'Zona restringida {idx+1}')
        fig.add_trace(go.Scatter3d(x=x,y=y,z=z,mode='lines',line=dict(color='#ff5578',width=5,dash='dash'),name=label,hovertemplate=f'<b>{label}</b><br>Radio: {zone["radius"]}°<extra></extra>'))
    # Target regions
    rx,ry,rz,labels,cols,hov=[],[],[],[],[],[]
    for name,info in REGIONS.items():
        x,y,z=latlon_to_xyz(*info['coords'],1.035); rx.append(x);ry.append(y);rz.append(z);labels.append(name)
        cols.append({3:'#ffc857',2:'#20d9f2',1:'#8ef0a4'}[info['priority']])
        prioridad_label = {3: 'Alta', 2: 'Media', 1: 'Normal'}[info['priority']]
        hov.append(f"{name} · prioridad {prioridad_label}")
    fig.add_trace(go.Scatter3d(x=rx,y=ry,z=rz,mode='markers+text',text=labels,textposition='top center',
        textfont=dict(size=10,color='#eafaff'),marker=dict(size=7,color=cols,symbol='diamond',line=dict(color='#ffffff',width=1)),
        name='Regiones objetivo',hovertext=hov,hovertemplate='%{hovertext}<extra></extra>'))
    sx,sy,sz,hovers=[],[],[],[]
    for i,sat in enumerate(individuo):
        x,y,z=latlon_to_xyz(sat[0],sat[1],1.075);sx.append(x);sy.append(y);sz.append(z)
        near=min(REGIONS,key=lambda n:satellite_region_distance(sat,n)); dist=satellite_region_distance(sat,near)
        hovers.append(f'<b>Satélite S{i+1:02}</b><br>Latitud: {sat[0]:.2f}°<br>Longitud: {sat[1]:.2f}°<br>Región cercana: {near}<br>Distancia: {dist:.1f} km<br>Posición válida: {"Sí" if is_valid_position(*sat) else "No"}')
    fig.add_trace(go.Scatter3d(x=sx,y=sy,z=sz,mode='markers+text',text=[f'S{i+1:02}' for i in range(len(individuo))],textposition='top center',
        textfont=dict(size=10,color='#8affea'),marker=dict(size=6,color=['#00f5d4' if is_valid_position(*s) else '#ff5578' for s in individuo],line=dict(color='#eaffff',width=1)),
        name='Satélites',hovertext=hovers,hovertemplate='%{hovertext}<extra></extra>'))
    # Coverage links for regions within observation radius
    for name,info in REGIONS.items():
        ds=[satellite_region_distance(s,name) for s in individuo]; idx=int(np.argmin(ds))
        if ds[idx] <= OBSERVATION_RADIUS_KM:
            a=latlon_to_xyz(*individuo[idx],1.065); b=latlon_to_xyz(*info['coords'],1.04)
            fig.add_trace(go.Scatter3d(x=[a[0],b[0]],y=[a[1],b[1]],z=[a[2],b[2]],mode='lines',line=dict(color='#00f5d4',width=4),opacity=.9,showlegend=False,hoverinfo='skip'))
    fig.update_layout(title=dict(text='GLOBO TÁCTICO · POSICIÓN DE SATÉLITES Y OBJETIVOS',font=dict(size=15,color='#dffaff')),
        paper_bgcolor='#07111f',plot_bgcolor='#07111f',font=dict(color='#dffaff',family='Arial'),
        legend=dict(bgcolor='rgba(6,20,34,.85)',bordercolor='#17445a',borderwidth=1,x=.02,y=.02,font=dict(size=10)),
        margin=dict(l=0,r=0,t=48,b=0),height=660,
        scene=dict(bgcolor='#07111f',xaxis=dict(visible=False,range=[-1.2,1.2]),yaxis=dict(visible=False,range=[-1.2,1.2]),zaxis=dict(visible=False,range=[-1.2,1.2]),aspectmode='cube',camera=dict(eye=dict(x=1.55,y=1.35,z=1.1))))
    display(HTML('<div style="font:12px Arial;color:#63a7bf;letter-spacing:2px;margin:14px 2px 5px">01 / VISTA PLANETARIA INTERACTIVA</div>'))
    fig.show()

    # Coverage report as a styled interactive dataframe
    styled = reporte.style.set_properties(**{'background-color':'#0b1d2b','color':'#e4f8ff','border-color':'#20445a','text-align':'left'}).set_table_styles([
        {'selector':'th','props':[('background-color','#12354a'),('color','#64f0dc'),('font-size','11px'),('text-transform','uppercase')]},
        {'selector':'caption','props':[('caption-side','top'),('color','#e4f8ff'),('font-size','16px'),('font-weight','bold'),('text-align','left'),('padding','10px 0')]}
    ]).set_caption('02 / ESTADO DE COBERTURA POR REGIÓN')
    display(styled)
    if historial_max is not None and historial_avg is not None:
        evo=go.Figure()
        evo.add_trace(go.Scatter(y=historial_max,mode='lines',name='Mejor fitness',line=dict(color='#00f5d4',width=3),fill='tozeroy',fillcolor='rgba(0,245,212,.08)'))
        evo.add_trace(go.Scatter(y=historial_avg,mode='lines',name='Fitness promedio',line=dict(color='#ffc857',width=2,dash='dot')))
        evo.update_layout(title=dict(text='03 / EVOLUCIÓN DEL ALGORITMO GENÉTICO',font=dict(size=15,color='#dffaff')),
            xaxis_title='Generación',yaxis_title='Fitness',paper_bgcolor='#07111f',plot_bgcolor='#0a1b2a',
            font=dict(color='#dffaff'),height=350,margin=dict(l=35,r=25,t=55,b=40),
            xaxis=dict(gridcolor='#18394b',zerolinecolor='#18394b'),yaxis=dict(gridcolor='#18394b',zerolinecolor='#18394b'),
            legend=dict(orientation='h',y=1.12,x=0))
        evo.show()
    display(HTML(f'<div style="font:11px Arial;color:#64869a;border-top:1px solid #18394b;padding-top:10px;margin-top:12px">ASTROSHIELD · FIN DE REPORTE · Cobertura calculada con radio de observación de {OBSERVATION_RADIUS_KM} km · Los resultados corresponden a la corrida ejecutada en este notebook.</div>'))

crear_dashboard(MEJOR_INDIVIDUO, HISTORIAL_MAX, HISTORIAL_AVG)


,Región,Prioridad,Distancia mínima (km),Satélite más cercano,Cubierta (≤300 km)
0,Yucatán,Alta,278.300000,6,Sí
1,Amazonía,Alta,93.700000,8,Sí
2,California,Media,201.500000,4,Sí
3,Sahel,Media,5483.400000,5,No
4,Japón,Media,280.100000,1,Sí
5,Australia,Normal,7008.100000,1,No


### Verificación básica de la solución y del reporte

In [7]:
assert len(MEJOR_INDIVIDUO) == 8, "La constelación debe tener 8 satélites."
assert len(coverage_report(MEJOR_INDIVIDUO)) == 6, "El reporte debe incluir 6 regiones."
assert all(-90 <= s[0] <= 90 and -180 <= s[1] <= 180 for s in MEJOR_INDIVIDUO)
print("Pruebas básicas completadas. Revisa el reporte para confirmar cobertura y restricciones.")


Pruebas básicas completadas. Revisa el reporte para confirmar cobertura y restricciones.


## Sección 4: Pruebas y Validación (Integrante 4)
Ejecución de pruebas iniciales, verificación de restricciones y análisis de múltiples corridas.

In [8]:
# Herramientas de prueba 
import copy, time, random
from contextlib import contextmanager
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Verifica que ya se ejecutaron las Secciones 1–3 (las pruebas usan sus funciones tal cual)
for _n in ("haversine", "is_valid_position", "evaluar_fitness", "coverage_report",
           "AlgoritmoEvolutivoConstelacion", "REGIONS", "DEBRIS_ZONES", "BLOCKED_OBJECTS",
           "OBSERVATION_RADIUS_KM"):
    if _n not in globals():
        raise NameError(f"Falta `{_n}`. Ejecuta primero las Secciones 1, 2 y 3 del notebook.")

R_TIERRA_KM = 6371.0

# Parámetros de las pruebas (mismos que usa la ejecución principal del equipo)
GEN_PRUEBA = 60        # generaciones por corrida
POB_PRUEBA = 50        # tamaño de población
MUT_PRUEBA = 0.12      # tasa de mutación
SEMILLAS = list(range(1, 11))   # 10 corridas para la prueba de variación

REGISTRO_PRUEBAS = []

def registrar(prueba, condiciones, resultado, paso, observacion=""):
    # Guarda el resultado de una prueba para la tabla resumen final.
    REGISTRO_PRUEBAS.append({"Prueba": prueba, "Condiciones": condiciones,
                             "Resultado": resultado, "¿Pasa?": "PASA" if paso else "FALLA",
                             "Observación": observacion})
    print(("✅ PASA" if paso else "❌ FALLA"), "—", prueba)

def distancia_grados(lat1, lon1, lat2, lon2):
    # Distancia angular exacta sobre la esfera (no usa la aproximación de 111 km/grado).
    return np.degrees(haversine(lat1, lon1, lat2, lon2) / R_TIERRA_KM)

def nombre_zona(z, i, prefijo):
    return z.get("name", f"{prefijo} {i + 1}")

def auditar(individuo):
    # Validador independiente: revisa cada satélite contra cada zona y cada región.
    invasiones = []
    for s_idx, (lat, lon) in enumerate(individuo):
        for tipo, lista, pref in (("Desechos", DEBRIS_ZONES, "Zona"),
                                  ("Objeto ocupado", BLOCKED_OBJECTS, "Objeto")):
            for i, z in enumerate(lista):
                d = distancia_grados(lat, lon, *z["center"])
                if d <= z["radius"]:
                    invasiones.append({"Satélite": s_idx + 1, "Tipo": tipo,
                                       "Zona": nombre_zona(z, i, pref),
                                       "Distancia (°)": round(d, 3), "Radio (°)": z["radius"]})
    cubiertas = [n for n, info in REGIONS.items()
                 if min(haversine(s[0], s[1], *info["coords"]) for s in individuo) <= OBSERVATION_RADIUS_KM]
    return {"n_satelites": len(individuo),
            "invasiones": invasiones,
            "valida": len(invasiones) == 0 and len(individuo) == 8,
            "cubiertas": cubiertas,
            "n_cubiertas": len(cubiertas)}

def margenes_satelites(individuo):
    # Tabla por satélite: zona prohibida más cercana y margen (positivo = fuera, negativo = invade).
    filas = []
    for s_idx, (lat, lon) in enumerate(individuo):
        mejor = None
        for tipo, lista, pref in (("Desechos", DEBRIS_ZONES, "Zona"),
                                  ("Objeto", BLOCKED_OBJECTS, "Objeto")):
            for i, z in enumerate(lista):
                margen = distancia_grados(lat, lon, *z["center"]) - z["radius"]
                if mejor is None or margen < mejor[2]:
                    mejor = (tipo, nombre_zona(z, i, pref), margen)
        filas.append({"Satélite": s_idx + 1, "Lat": round(lat, 2), "Lon": round(lon, 2),
                      "Restricción más cercana": f"{mejor[0]}: {mejor[1]}",
                      "Margen (°)": round(mejor[2], 2),
                      "is_valid_position": is_valid_position(lat, lon)})
    return pd.DataFrame(filas)

@contextmanager
def escenario_temporal(regiones=None, desechos=None, objetos=None):
    # Cambia temporalmente las globales de la Sección 1 y SIEMPRE las restaura al final.
    g = globals()
    respaldo = (copy.deepcopy(REGIONS), copy.deepcopy(DEBRIS_ZONES), copy.deepcopy(BLOCKED_OBJECTS))
    try:
        if regiones is not None: g["REGIONS"] = regiones
        if desechos is not None: g["DEBRIS_ZONES"] = desechos
        if objetos is not None: g["BLOCKED_OBJECTS"] = objetos
        yield
    finally:
        g["REGIONS"], g["DEBRIS_ZONES"], g["BLOCKED_OBJECTS"] = respaldo

def correr_ae(semilla, generaciones=GEN_PRUEBA, poblacion=POB_PRUEBA, mutacion=MUT_PRUEBA):
    # Ejecuta el algoritmo del Integrante 2 con semilla fija (reproducible) y audita el resultado.
    random.seed(semilla); np.random.seed(semilla)
    ae = AlgoritmoEvolutivoConstelacion(evaluar_fitness_func=evaluar_fitness,
                                        validar_posicion_func=is_valid_position, num_satelites=8)
    t0 = time.time()
    mejor, fit, h_max, h_avg = ae.ejecutar(generaciones=generaciones, tamano_poblacion=poblacion,
                                           tasa_mutacion=mutacion)
    return {"semilla": semilla, "mejor": [list(s) for s in mejor], "fitness": fit,
            "hist_max": h_max, "hist_avg": h_avg, "tiempo_s": time.time() - t0,
            "auditoria": auditar(mejor)}

print("Herramientas de prueba listas.")

Herramientas de prueba listas.


In [9]:
# ===== Prueba 0: funciones base =====
casos = []
# Haversine contra valores conocidos
d_1grado = haversine(0, 0, 0, 1)
casos.append(("1° de longitud en el ecuador ≈ 111.19 km", abs(d_1grado - 111.19) < 0.1, f"{d_1grado:.2f} km"))
d_antip = haversine(0, 0, 0, 180)
casos.append(("Puntos antípodas ≈ 20015 km", abs(d_antip - 20015.1) < 1, f"{d_antip:.1f} km"))
casos.append(("Distancia simétrica", abs(haversine(20.5, -89, 36, 138) - haversine(36, 138, 20.5, -89)) < 1e-9, "ok"))
casos.append(("Distancia a sí mismo = 0", haversine(15, 0, 15, 0) == 0, "0 km"))

# is_valid_position: centros de zonas deben ser inválidos
for z in DEBRIS_ZONES + BLOCKED_OBJECTS:
    casos.append((f"Centro {z['center']} es inválido", not is_valid_position(*z["center"]), ""))
# Un punto lejos de todo (Pacífico central) debe ser válido
casos.append(("Punto libre (0, -150) es válido", is_valid_position(0, -150), ""))

df0 = pd.DataFrame(casos, columns=["Caso", "Correcto", "Valor"])
display(df0)

# Caso borde: punto a 99.9 % del radio del Atlántico Norte (está DENTRO de la zona)
z = DEBRIS_ZONES[0]
lat_borde = z["center"][0] + z["radius"] * 0.999   # mismo meridiano → distancia = 11.988°
dentro_real = distancia_grados(lat_borde, z["center"][1], *z["center"]) <= z["radius"]
acepta = is_valid_position(lat_borde, z["center"][1])
print(f"\nPunto borde a {z['radius']*0.999:.3f}° del centro → dentro de la zona: {dentro_real}; "
      f"is_valid_position lo acepta: {acepta}")

paso0 = bool(df0["Correcto"].all())
obs0 = ("is_valid_position usa 111 km/° (el valor exacto es 111.19 km/°): acepta puntos que están "
        "hasta ~0.2 % dentro del borde de cada zona." if acepta and dentro_real else "Borde correcto.")
registrar("P0 Funciones base", "Distancias conocidas, centros y borde de zonas",
          f"{int(df0['Correcto'].sum())}/{len(df0)} casos correctos", paso0, obs0)

,Caso,Correcto,Valor
0,1° de longitud en el ecuador ≈ 111.19 km,True,111.19 km
1,Puntos antípodas ≈ 20015 km,True,20015.1 km
2,Distancia simétrica,True,ok
3,Distancia a sí mismo = 0,True,0 km
4,"Centro (45.0, -35.0) es inválido",True,
5,"Centro (-30.0, -120.0) es inválido",True,
6,"Centro (78.0, 30.0) es inválido",True,
7,"Centro (10.0, 45.0) es inválido",True,
8,"Centro (-15.0, 110.0) es inválido",True,
9,"Punto libre (0, -150) es válido",True,



Punto borde a 11.988° del centro → dentro de la zona: True; is_valid_position lo acepta: True
✅ PASA — P0 Funciones base


**Interpretación P0:** 10/10 casos correctos. Haversine y la detección de centros de zona funcionan. El caso borde muestra que `is_valid_position` convierte grados a km con 111 km/° cuando el valor exacto es 2πR/360 ≈ 111.19 km/°. Por eso hay una franja delgada (≈ 0.17 % del radio, unos 2 km en el Atlántico Norte) donde un satélite **dentro** de la zona se acepta como válido. El efecto práctico es mínimo, pero es un error de la restricción y se reporta al Integrante 1. Corrección sugerida: comparar contra `radius * math.pi * 6371 / 180`.

## Sección 5: Análisis de Resultados y Conclusiones (Integrante 5)
> **Definición del problema y análisis técnico:**
> * Las restricciones espaciales se aplicaron mediante penalizaciones matemáticas en la función de fitness.
> * El algoritmo logra explorar configuraciones válidas evitando los puntos prohibidos de desechos espaciales.